# Учись Машина Учись / Learn Machine Learn

<img src="data/lml.png" width=200>

Онлайн-лекции Ильи С. Елисеева: применение методов машинного обучения в анализе данных.

- Канал в Telegram: https://t.me/learn_machine_learn
- YouTube: https://www.youtube.com/channel/UCCwDwKatNitBCVAJajremMQ
- VK: https://vk.com/learn_machine_learn
- GitHub: https://github.com/easyise/learn_machine_learn

---



## 2. Проблемы в данных GPS треков: "телепорты" и ко.

1. **Выбросы-телепорты**. Скачки GPS сигнала, когда устройство внезапно "перемещается" на большое расстояние за короткий промежуток времени.
2. **"Я столько не выжму"** - невероятные/недостижимые скорости или ускорения. Такие значения можно задать эмпирически или получить статистическим путем через z-статистику.
3. **Прерывающийся трек**: остановился чаю попить, поставил запись трека на паузу и забыл снять - знакомая ситуация? Вспомнил об этом только спустя несколько километров - что делать? 
4. **Первое, второе и третье вместе взятое**.

Что делать:
1. Детекция и удаление выбросов-телепортов. По возможности, аппроксимация пропущенных точек по нитке маршрута.
2. Фильтрация точек с нереалистичными скоростями и ускорениями - иногда работает простое удаление, оно же сглаживает трек и устраняет мелкие скачки.
3. Вычисление триангулярной скорости:
    - скорость прибытия в точку (от предыдущей точки к текущей)
    - скорость убытия из точки (от текущей точки к следующей)
    - скорость передвижения от предыдущей к следующей точке (без учета текущей)

    Если скорость по основанию треугольника сильно отличается от прибытия или убытия, то точка может быть выбросом.

2. Фильтрация с помощью алгоритмов кластеризации, например DBSCAN.
3. Обработка прерывающегося трека: сегментация трека на отдельные части.

In [ ]:
%pip install gpxpy geopy geopandas folium contextily ipywidgets

In [ ]:
import pandas as pd
import geopandas as gpd

import numpy as np
import matplotlib.pyplot as plt

import contextily as cx 

%load_ext autoreload
%autoreload 2

from utils.geodata import *

In [ ]:
gdf_ski = gpd.read_file("data/Afternoon_Backcountry_Ski.gpx", 
                        layer="track_points")
gdf_ski

In [ ]:
gdf_ski = recalculate_metrics(gdf_ski)
gdf_ski

In [ ]:
# превышена третья космическая скорость (60120 км/ч)!
gdf_ski[['time', 'timedelta_s', 'dist', 'speed_kmh', 'acc_m_per_s2', 'acc_g']].describe()

In [ ]:
ax = get_with_segments(gdf_ski)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="turbo",
        vmin=0,
        vmax=60,
        legend=True,
        figsize=(8, 8),
)

ax.set_title("GPX track colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

### 2.1 Пропуски и остановки

Явные остановки определяются по признаку "расстояние == 0". Такие данные можно не рассматривать. 

Но в каких случаях их лучше оставить? 

In [ ]:
print("Всего остановок:", (gdf_ski["dist"] == 0).sum())

### 2.2 Поиск аномалий в скоростях, ускорениях и т.д.

Выполним расчеты скоростей и ускорений, чтобы затем отфильтровать точки с нереалистичными значениями.

In [ ]:
recalculate_metrics?

In [ ]:
gdf_ = recalculate_metrics(gdf_ski)
gdf_[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2', 'acc_g', 'prev_point', 'geometry']].head(20)

In [ ]:
gdf_[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2', 'acc_g', 'geometry', 'prev_point']].describe()

In [ ]:
import seaborn as sns

bins = 100

fig, axs = plt.subplots(1, 3, figsize=(12, 4))

axs[0].set_title("Distance (m)")
axs[1].set_title("Speed (km/h)")
axs[2].set_title("Acceleration (m/s²)")

sns.histplot(gdf_["dist"]+1, bins=bins, kde=True, ax=axs[0], log_scale=True)
sns.histplot(gdf_["speed_kmh"]+1, bins=bins, kde=True, ax=axs[1], log_scale=True)
sns.histplot(gdf_["acc_m_per_s2"], bins=bins, kde=True, ax=axs[2]);

Количественный анализ выбросов: эмпирически и на основе теории вероятностей.

Эмпирический способ: установить порог на основе здравого смысла. Например, если человек не может двигаться быстрее 100 км/ч, то все точки с такой скоростью и выше можно считать выбросами.

In [ ]:
vmax = 60 # km/h

print("Всего сегментов с превышением скорости {} км/ч: {}".format(vmax, (gdf_['speed_kmh'] > vmax).sum()))

На основе теории вероятностей: правило Тьюки.



In [ ]:
from utils.plotting import plot_outliers

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

plot_outliers(gdf_["dist"], ax=axs[0], method='tukey', bins=100, logscale=True)
plot_outliers(gdf_["speed_kmh"], ax=axs[1], method='tukey', bins=100, logscale=True)

На основе теории вероятностей: z-статистика. 

Что такое z? Это количество стандартных отклонений, на которое значение отличается от среднего.

In [ ]:
from ipywidgets import interact

# нормально распределенные данные:
norm_data = np.random.normal(loc=0, scale=1, size=10000)

z_theshs = (1, 5)

def plot_with_z_thres(z_thres):
    plt.figure(figsize=(8, 4))
    plot_outliers(pd.Series(norm_data), ax=plt.gca(), method='z-score', z_thres=z_thres, bins=30)
    plt.title("Normally Distributed Data")
    plt.xlabel("Value")
    plt.ylabel("Frequency")
    plt.show()

    
interact(plot_with_z_thres, z_thres=z_theshs);


Но с GPS данными все по-другому! Они не распределены нормально. Поэтому обычная z-статистика может не работать, так как она предполагает нормальное распределение данных.

Здесь приходится пользоваться z-статистикой, основанной на медиане и MAD (Median Absolute Deviation). Это так называемая "**робастная**" z-статистика, которая не чувствительна к выбросам и может работать с данными, имеющими тяжелые хвосты.

In [ ]:
z_thres = 3 # обычно 3, но можно брать больше


def get_z_scores(gdf, z, cols=['speed_kmh', 'acc_m_per_s2']):
    ret_thres = tuple()
    for col in cols:
        v = gdf[col]
        median = v.median()
        mad = (v - median).abs().median()
        ret_thres += (median + z * 1.4826 * mad,)
        gdf[f'{col}_z'] = np.abs(0.6745 * (v - median) / mad)
        gdf[f'{col}_z_fail'] = gdf[f'{col}_z'] > z
        gdf[f'{col}_z'] = gdf[f'{col}_z'].fillna(0)
        gdf[f'{col}_z_fail'] = gdf[f'{col}_z_fail'].fillna(False)
    return (gdf,) + ret_thres

gdf_, speed_thres, acc_thres = get_z_scores(gdf_, z=z_thres)
print("Всего сегментов с превышением z={} для скорости {:.2f} км/ч: {}".format(z_thres, speed_thres, (gdf_['speed_kmh_z_fail']).sum()))
print("Всего сегментов с превышением z={} для ускорения {:.2f} м/с²: {}".format(z_thres, acc_thres, (gdf_['acc_m_per_s2_z_fail']).sum()))

In [ ]:
def plot_outliers_with_z(col, z_thres=3):
    fig, ax = plt.subplots(figsize=(8, 4))
    gdf, thres = get_z_scores(gdf_, z_thres, cols=[col])
    sns.histplot(gdf[col]+1, bins=100, kde=True, ax=ax, log_scale=True)
    ax.axvline(gdf[col].median(), color='r', linestyle='--', label=f'Медиана: {gdf[col].median():.2f}')
    ax.axvline(thres, color='r', linestyle='--', label=f'Z-порог: {thres:.2f}')
    ax.axvspan(thres, gdf[col].max(), alpha=0.1, color='red', label=f'Выбросы: {(gdf[col] > thres).sum()}')
    ax.legend()
    ax.set_title(f"{col} with z-score outliers")

z_theshs = (1, 7)
col = ['speed_kmh', 'acc_m_per_s2']

interact(plot_outliers_with_z, col=col, z_thres=z_theshs);


### 2.3. Фильтрация по значениям скорости в треугольнике

Рассматривается треугольник, образованный тремя последовательными точками маршрута. Вычисляются скорости по каждой из сторон треугольника, и если скорость по основанию сильно отличается от скоростей по боковым сторонам, то центральная точка может быть выбросом.

In [ ]:
gdf_

In [ ]:
# добавим еще метрики триангуляции
def recalculate_triangulation_metrics(gdf_):
    
    gdf_['next_point'] = gdf_.geometry.shift(-1)
    
    # базис треугольника (расстояние между предыдущей и следующей точками)
    gdf_['dist_base'] = gdf_.apply(lambda row: haversine(
        row.prev_point.y if pd.notna(row.prev_point) else row.geometry.y,
        row.prev_point.x if pd.notna(row.prev_point) else row.geometry.x,
        row.next_point.y if pd.notna(row.next_point) else row.geometry.y,
        row.next_point.x if pd.notna(row.next_point) else row.geometry.x
    ), axis=1)

    # Calculate speeds out and base in km/h
    gdf_['speed_out'] = (gdf_['dist'].shift(-1) / gdf_['timedelta_s'].shift(-1) * 3.6).fillna(0)
    gdf_['speed_base'] = (gdf_['dist_base'] / (gdf_['timedelta_s'] + gdf_['timedelta_s'].shift(-1)) * 3.6).fillna(0)

    # Коэффициент различия между скоростями в основании и по сторонам треугольника. 
    # Если скорость в основании значительно ниже, чем по сторонам, то точка может быть выбросом.
    # Принимается значение менее 0.8 как подозрительное.
    max_speed = gdf_[['speed_kmh', 'speed_out']].max(axis=1)
    gdf_['triang_diff'] = (gdf_['speed_base'] / max_speed).fillna(1)

    return gdf_

recalculate_triangulation_metrics(gdf_)
gdf_[['time', 'speed_kmh', 'speed_out', 'speed_base', 'triang_diff']]

In [ ]:
# вычислим количество проблемных точек по триангуляции
threshold_speed = 30  # km/h

triang_mask = ( ((gdf_['speed_kmh'] > threshold_speed) |
                (gdf_['speed_out'] > threshold_speed)) &
                (gdf_['triang_diff'] < 0.8) )

print("Количество проблемных точек по триангуляции с порогом {} км/ч: {}".format(threshold_speed, triang_mask.sum()))

gdf_[['time', 'speed_kmh', 'speed_out', 'speed_base', 'triang_diff']][triang_mask]

### 2.4 Анализ трека как временного ряда

In [ ]:
# Временной ряд
gdf_ts = gdf_.set_index('time').sort_index()

fig, ax = plt.subplots(figsize=(12, 4))

gdf_ts['speed_kmh'].plot(ax=ax);
# ax.set_xlim('2026-01-18 13:26', '2026-01-18 13:38')
# ax.set_ylim(0, 200)
ax.grid()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

gdf_ts['acc_m_per_s2'].plot(ax=ax);
# ax.set_xlim('2026-01-18 13:26', '2026-01-18 13:38')
# ax.set_ylim(0, 200)
ax.grid()
plt.show()

In [ ]:
fig, axs = plt.subplots( 3, 1, figsize=(12, 12))

gdf_ts_disp = gdf_ts['2026-01-18 13:26':'2026-01-18 13:38']
# gdf_ts_disp = gdf_ts

gdf_ts_disp['speed_kmh'].plot(ax=axs[0], title='Speed (km/h)')
gdf_ts_disp['acc_m_per_s2'].plot(ax=axs[1], title='Acceleration (m/s²)')
gdf_ts_disp['triang_diff'].plot(ax=axs[2], title='Triangulation Difference')

for ax in axs:
    ax.grid()

fig.tight_layout()
plt.show()

### 2.5 Анализ скученности проблемных точек

```

... норм — норм — ❌ — норм — норм ...

```

или
```

... норм — ❌ — ❌ — ❌ — норм ...

```

In [ ]:
def run_lengths(mask: pd.Series):
    """
    Возвращает DataFrame с сериями подряд True:
    start_idx, end_idx, length
    """
    mask = mask.fillna(False).astype(bool)

    # меняется ли значение по сравнению с предыдущим
    grp = (mask != mask.shift()).cumsum()

    runs = (
        mask
        .groupby(grp)
        .agg(
            value="first",
            length="size",
            start_idx=lambda s: s.index[0],
            end_idx=lambda s: s.index[-1],
        )
        .reset_index(drop=True)
    )

    # оставляем только True-серии
    runs = runs[runs["value"]].drop(columns="value")
    return runs

v_max = 30 # km/h - допустим, я больше не выжму на лыжах

runs_speed_ok = run_lengths(gdf_['speed_kmh'] > v_max) 
runs_speed_ok  = runs_speed_ok.sort_values(by='length', ascending=False)
runs_speed_ok.head(10)

In [ ]:
gdf_.loc[runs_speed_ok.iloc[0]['start_idx']:runs_speed_ok.iloc[0]['end_idx']]['speed_kmh']

### 2.6 Кластерный анализ трека

Кластерный анализ позволит не только определить выбросы, но и выполнить сегментацию трека в случае его прерываний.

Кластеризацию будем выполнять только для пространственных данных, т.е. точек, из которых состоит трек. 

Для кластеризации будет использоваться алгоритм ```DBSCAN``` из библиотеки scikit-learn и ```HDBSCAN``` из библиотеки hdbscan.

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) - алгоритм кластеризации, который группирует точки на основе плотности их распределения в пространстве.

HDBSCAN (Hierarchical Density-Based Spatial Clustering of Applications with Noise) - расширение DBSCAN, которое использует иерархический подход для определения кластеров на основе плотности точек.

In [ ]:

# используем DBSCAN для поиска выбросов по скорости и ускорению
from sklearn.cluster import DBSCAN

# подбираем CRS с максимально подходящей проекцией
utm_crs = gdf_.estimate_utm_crs()
gdf_utm = gdf_.to_crs(utm_crs)

# формируем матрицу признаков из метрических координат
X = np.column_stack([gdf_utm.geometry.x.to_numpy(), gdf_utm.geometry.y.to_numpy()])

X



In [ ]:
# DBSCAN
# eps = радиус соседства (метры). Пример: 30м для плотных GPS-точек.
# min_samples = минимум точек в кластере (пример: 10)
eps_m = 30  # метров
min_samples = 10  # минимум точек в кластере (пример: 5 минут при 1 Гц)

labels = DBSCAN(eps=eps_m, min_samples=min_samples).fit_predict(X)
gdf_['dbscan_label'] = labels
gdf_['dbscan_label'] = gdf_['dbscan_label'].astype('category')
gdf_['dbscan_label'].value_counts()

In [ ]:
ax = gdf_[['time', 'geometry', 'dbscan_label']] \
    .to_crs(epsg=3857) \
    .plot(
        column="dbscan_label",
        cmap="turbo",
        legend=True,
        alpha=0.9,
        markersize=2,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by DBSCAN labels")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1)

ax.grid()
plt.show()

In [ ]:
ax = gdf_[['time', 'geometry', 'dbscan_label']] \
    .to_crs(epsg=3857) \
    .plot(
        column="dbscan_label",
        cmap="turbo",
        legend=True,
        alpha=0.9,
        markersize=2,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by DBSCAN labels")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom=14)

targe_cluster = 6

bounds = np.array(gdf_[gdf_.dbscan_label==targe_cluster].to_crs(epsg=3857).total_bounds)
margin = 200
bounds[[0,1]] -= margin
bounds[[2,3]] += margin

ax.set_ylim(bounds[[1, 3]])
ax.set_xlim(bounds[[0, 2]])

ax.grid()
plt.show()

In [ ]:
# как подобрать оптимально параметры DBSCAN

from sklearn.neighbors import NearestNeighbors

def choose_eps_knee(X, k=10, quantile=0.98):
    """
    X: (N,2) координаты в метрах (например, UTM)
    k: min_samples
    Возвращает eps по "колену" (упрощённо: высокий квантиль k-distance).
    """
    nn = NearestNeighbors(n_neighbors=k).fit(X)
    dists, _ = nn.kneighbors(X)
    kdist = np.sort(dists[:, -1])  # расстояние до k-го соседа

    # Простой устойчивый автопорог: почти-колено ≈ верхний квантиль плотной зоны
    eps = float(np.quantile(kdist, quantile))
    return eps, kdist

# --- пример применения ---
# gdf -> UTM -> X
# utm = gdf.estimate_utm_crs(); gdf_m = gdf.to_crs(utm)
# X = np.c_[gdf_m.geometry.x.to_numpy(), gdf_m.geometry.y.to_numpy()]

min_samples = 5 * 60
eps, kdist = choose_eps_knee(X, k=min_samples, quantile=0.98)

labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)

print(f"Выбранный eps: {eps:.2f} м")

gdf_['dbscan_label_auto'] = labels
gdf_['dbscan_label_auto'] = gdf_['dbscan_label_auto'].astype('category')
gdf_['dbscan_label_auto'].value_counts()

In [ ]:
ax = gdf_[['time', 'geometry', 'dbscan_label_auto']] \
    .to_crs(epsg=3857) \
    .plot(
        column="dbscan_label_auto",
        cmap="turbo",
        legend=True,
        alpha=0.9,
        markersize=2,
        figsize=(10, 10),
)

ax.set_title("GPX track colored by DBSCAN labels")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)

ax.grid()
plt.show()

In [ ]:
%pip install hdbscan

In [ ]:
# выполним анализ кластеров с помощью HDBSCAN
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5 * 60,  # минимум точек в кластере (пример: 5 минут при 1 Гц)
    # min_samples=10,   # можно не задавать
    metric="euclidean"
)

labels = clusterer.fit_predict(X)
gdf_["hdbscan_label"] = labels
# gdf_["hdbscan_label"] = gdf_["hdbscan_label"].astype("category")
gdf_["hdbscan_label"].value_counts()

In [ ]:
clusterer.probabilities_

In [ ]:
gdf_["hdbscan_proba"] = clusterer.probabilities_
gdf_[["hdbscan_label", "hdbscan_proba"]]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 6))

mask = gdf_['hdbscan_label'] == 1
gdf_disp = gdf_[mask]

gdf_disp[['time', 'geometry', 'hdbscan_label']] \
    .to_crs(epsg=3857) \
    .plot(
        column="hdbscan_label",
        cmap="viridis",
        legend=True,
        alpha=0.9,
        markersize=2,
        ax = axs[0]
)
gdf_disp[['time', 'geometry', 'hdbscan_proba']] \
    .to_crs(epsg=3857) \
    .plot(
        column="hdbscan_proba",
        cmap="viridis",
        legend=True,
        alpha=0.9,
        markersize=2,
        ax = axs[1]
)

fig.suptitle("GPX track colored by HDBSCAN labels")
for ax in axs:
    cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
    ax.grid()

plt.show()

In [ ]:
# полная кластеризация: добавим время и скорость в признаки
from sklearn.preprocessing import StandardScaler
def clustering_features(gdf_, include=['time', 'dist', 'speed_kmh','acc_m_per_s2']):
    gdf_ = gdf_.copy()
    
    # подбираем CRS с максимально подходящей проекцией
    utm_crs = gdf_.estimate_utm_crs()
    gdf_utm = gdf_.to_crs(utm_crs)

    # формируем матрицу признаков из метрических координат + времени + скорости
    coords_x = gdf_utm.geometry.x.to_numpy()
    coords_y = gdf_utm.geometry.y.to_numpy()
    list_of_features = [coords_x, coords_y]
    if 'time' in include:
        time_s = (gdf_.time - gdf_.time.iloc[0]).dt.total_seconds().to_numpy()
        list_of_features.append(time_s)
    if 'dist' in include:
        dist_m = gdf_['dist'].to_numpy()
        list_of_features.append(dist_m)
    if 'speed_kmh' in include:
        speed_kmh = gdf_['speed_kmh'].to_numpy()
        list_of_features.append(speed_kmh)
    if 'acc_m_per_s2' in include:
        acc_m_per_s2 = gdf_['acc_m_per_s2'].to_numpy()
        list_of_features.append(acc_m_per_s2)

    X = np.column_stack(list_of_features)

    # стандартизируем признаки
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X_scaled

X_scaled = clustering_features(gdf_, include=['time', 'dist', 'speed_kmh'])
eps, kdist = choose_eps_knee(X_scaled, k=min_samples, quantile=0.98)

labels = DBSCAN(
    eps=eps,
    min_samples=5 * 60,
    metric="euclidean"
).fit_predict(X_scaled)
gdf_["cluster_full_label"] = labels
print(f"Выбранный eps для полной кластеризации, сигм: {eps:.2f}")
gdf_["cluster_full_label"].value_counts()

In [ ]:
gdf_bike_bad = gpd.read_file('data/Bike_bad.gpx', 
                        layer="track_points")
recalculate_metrics(gdf_bike_bad)
display(gdf_bike_bad)

ax = get_with_segments(gdf_bike_bad)[['time', 'geometry', 'segment']] \
    .to_crs(epsg=3857) \
    .dropna() \
    .plot(figsize=(8, 8), linewidth=0.5, color='blue')
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
def get_cluster_labels(gdf, min_samples=600, include=['time', 'dist', 'speed_kmh']):
    
    recalculate_metrics(gdf)

    X = clustering_features(gdf, include=include)

    eps_m, kdist = choose_eps_knee(X, k=min_samples, quantile=0.98)

    labels = DBSCAN(eps=eps_m, 
                    min_samples=min_samples,
                    metric="euclidean").fit_predict(X)
    return labels, eps_m

In [ ]:
labels, eps = get_cluster_labels(gdf_bike_bad, min_samples=2*60)
gdf_bike_bad["cluster_full_label"] = labels
gdf_bike_bad["cluster_full_label"] = gdf_bike_bad["cluster_full_label"].astype("category")
n_clusters = np.unique(labels[labels != -1]).size
noise_points = (labels == -1).sum()
print(f"Найдено кластеров: {n_clusters}, шумовых точек: {noise_points}, eps: {eps:.2f}")

ax = gdf_bike_bad[['time', 'geometry', 'cluster_full_label']] \
    .to_crs(epsg=3857) \
    .plot(
        column="cluster_full_label",
        cmap="tab10",
        legend=True,
        alpha=1,
        markersize=2,
        figsize=(10, 10),
)

ax.set_title("Clustering labels: {} clusters".format(n_clusters))
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1)

ax.grid()
plt.show()

### 2.7 Требует ли трек очистки от выбросов?

Как автоматически определить, что трек требует очистки от выбросов?

In [ ]:
# загрузим четыре трека:
gdf_ski_ok = gpd.read_file('data/S_Novym_Godom_Ski.gpx', 
                        layer="track_points")
gdf_ski_bad = gpd.read_file('data/Afternoon_Backcountry_Ski.gpx', 
                        layer="track_points")
gdf_bike_ok = gpd.read_file('data/Bike_ok.gpx', 
                        layer="track_points")
gdf_bike_bad = gpd.read_file('data/Bike_bad.gpx', 
                        layer="track_points")



In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(15, 15))

gdfs_ = [gdf_ski_ok, gdf_ski_bad,
          gdf_bike_ok, gdf_bike_bad]
titles = ['⛷️ Ski ok', '⛷️ Ski with outliers',
          '🚴 Bike ok', '🚴 Bike with outliers']

for i, (ax, gdf_) in enumerate(zip(axs.flatten(), gdfs_)):
    get_with_segments(gdf_)[['time', 'segment']] \
        .to_crs(epsg=3857) \
        .plot(
            alpha=0.9,
            linewidth=5,
            ax=ax
        )
    ax.set_title(titles[i][2:]+" points: {}".format(len(gdf_)))
    cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
    ax.grid()
    


In [ ]:
# рассчитаем метрики для всех четырёх треков
for i, gdf_ in enumerate(gdfs_):
    recalculate_metrics(gdf_)
    print(f"Метрики для трека: {titles[i]}")
    display(gdf_[['dist', 'speed_kmh', 'acc_m_per_s2']].describe())

In [ ]:
# выполним расчет робастных показателей
z_thresh = 6

def get_z_scores(gdf, z):
    ret_thres = tuple()
    for col in ['speed_kmh', 'acc_m_per_s2']:
        v = gdf[col]
        median = v.median()
        mad = (v - median).abs().median()
        ret_thres += (median + z * 1.4826 * mad,)
        gdf[f'{col}_z'] = np.abs(0.6745 * (v - median) / mad)
        gdf[f'{col}_z_fail'] = gdf[f'{col}_z'] > z
        gdf[f'{col}_z'] = gdf[f'{col}_z'].fillna(0)
        gdf[f'{col}_z_fail'] = gdf[f'{col}_z_fail'].fillna(False)
    return (gdf,) + ret_thres

for i, gdf_metrics in enumerate(gdfs_):
    print(f"Track {titles[i]}")
    gdf_metrics_z, v_rob, a_rob = get_z_scores(gdf_metrics, z_thresh)
    bad_z_speed = gdf_metrics_z['speed_kmh_z_fail'].sum()
    print('Speed failures z-test (>{:.2f} km/h): {} of {} ({:.2f}%)'.format(
                        v_rob,
                        bad_z_speed,
                        len(gdf_metrics_z), 
                        gdf_metrics_z['speed_kmh_z_fail'].mean() * 100,
                        ))
    bad_z_acc = gdf_metrics_z['acc_m_per_s2_z_fail'].sum()
    print('Acceleration failures z-test: {} of {} ({:.2f}%)'.format(
                        bad_z_acc,
                        len(gdf_metrics_z), 
                        gdf_metrics_z['acc_m_per_s2_z_fail'].mean() * 100,
                        ))
    print()


In [ ]:
# Делаем кластерный анализ
for i, gdf_metrics in enumerate(gdfs_):
    print(f"Track {titles[i]}")
    labels, eps_m = get_cluster_labels(gdf_metrics, min_samples=5*60)
    gdf_metrics['dbscan_label'] = labels
    n_clusters = np.unique(labels).size - (1 if -1 in np.unique(labels) else 0)
    n_noise = (labels == -1).sum()
    print(f"DBSCAN found {n_clusters} clusters, noise points: {n_noise}, eps: {eps_m:.2f} sigmas")
    print()


In [ ]:
# итак, пишем функцию, которая решает - нужно ли очищать трек от выбросов или нет
def is_cleanup_needed(gdf, z_thresh=6, speed_fail_ratio_thresh=0.05, acc_fail_ratio_thresh=0.05, min_dbscan_noise_ratio=0.1):
    gdf_metrics = recalculate_metrics(gdf)
    gdf_metrics_z, v_rob, a_rob = get_z_scores(gdf_metrics, z_thresh)
    
    speed_fail_ratio = gdf_metrics_z['speed_kmh_z_fail'].mean()
    acc_fail_ratio = gdf_metrics_z['acc_m_per_s2_z_fail'].mean()
    
    labels, eps_m = get_cluster_labels(gdf_metrics, min_samples=5*60)
    gdf_metrics['dbscan_label'] = labels
    n_clusters = np.unique(labels).size - (1 if -1 in np.unique(labels) else 0)
    n_noise = (labels == -1).sum()
    dbscan_noise_ratio = n_noise / labels.size

    cleanup_needed = (
        ((speed_fail_ratio > speed_fail_ratio_thresh) or
        (acc_fail_ratio > acc_fail_ratio_thresh)) and
        (n_clusters > 1 or
        (dbscan_noise_ratio > min_dbscan_noise_ratio)
        )
    )
    
    return cleanup_needed, {
        'speed_fail_ratio': speed_fail_ratio,
        'acc_fail_ratio': acc_fail_ratio,
        'dbscan_n_clusters': n_clusters,
        'dbscan_noise_ratio': dbscan_noise_ratio,
        'v_rob': v_rob,
        'a_rob': a_rob,
        'eps_m': eps_m
    }

for i, gdf_ in enumerate(gdfs_):
    print(f"Track {titles[i]}")
    cleanup_needed, metrics = is_cleanup_needed(gdf_,
                                               z_thresh=6,
                                               speed_fail_ratio_thresh=0.05,
                                               acc_fail_ratio_thresh=0.05,
                                               min_dbscan_noise_ratio=0.1)
    print("Cleanup needed:" , cleanup_needed)
    print("Metrics:", {k: round(float(v), 2) for k, v in metrics.items()})
    print()